# Producer A — Camera Event Stream (Camera 1)

This notebook implements the Kafka producer for **Camera 1** of the AWAS traffic monitoring pipeline. It reads `camera_event_A.csv`, groups records into batches by `batch_id`, and publishes each batch to the Kafka topic `camera-events-A` at a controlled rate to simulate a live roadside camera stream.

---

## Role in the Pipeline

Producer A simulates an independent roadside camera feed by continuously publishing Camera 1 events to Kafka. These streamed events are later consumed by Spark Structured Streaming for violation detection and cross-camera joins.

camera_event_A.csv → [Producer A] → Kafka topic: camera-events-A → Spark Structured Streaming

---

## EDA Findings

| Check | Finding |
|---|---|
| Camera IDs | Camera 1 only |
| Timestamp ordering | Not ordered within batches |
| Null speed readings | None found |
| Duplicate `car_plate` | 147 duplicates in vehicle.csv |
| Shared A/B plates | 9,844 matches with Producer B |
| `batch_id` | Local to Producer A (1–27,817) |

These findings justified using Spark watermarks and time-window joins rather than batch_id matching.

---

## Event Payload

Each Kafka message contains:

- `event_id` — unique camera event identifier
- `batch_id` — batch grouping key (local to Producer A)
- `car_plate` — vehicle registration plate (whitespace-stripped)
- `camera_id` — always `1` for this producer
- `timestamp` — original event time when the vehicle passed Camera 1
- `speed_reading` — recorded speed in km/h, rounded to 2 dp
- `producer_id` — always `'A'`
- `produced_at` — UTC ingestion timestamp added at publish time
- `batch_size` — total events in this batch for consumer completeness checks

`timestamp` stores the original event time; `produced_at` stores Kafka ingestion time. Spark uses `timestamp` for watermarking and joins.

---

## Key Parameters

| Parameter | Value | Purpose |
|---|---|---|
| `BATCH_SLEEP_SECS` | `2` | Simulates streaming cadence |
| `TOPIC` | `camera-events-A` | Dedicated Kafka topic for Camera 1 |
| `api_version` | `(2, 5, 0)` | Modern Kafka protocol |
| Error handling | `try/except` | Skips malformed rows without crashing |

---

## batch_id Behaviour

`batch_id` groups events into publishing batches and controls the publishing rate. Producer A has **27,817 unique batch IDs** — far fewer than B (294,206) or C (414,871), meaning each A batch contains more events on average (~20 per batch). Since batch IDs are not shared across producers, Spark joins use `car_plate` and event-time windows instead.

In [1]:
# Use %pip to ensure the package is linked to this specific notebook kernel
%pip install kafka-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import json
import time
from datetime import datetime, timezone
from collections import defaultdict
from kafka import KafkaProducer        
from kafka.errors import KafkaError 

print("Kafka library loaded successfully!")

Kafka library loaded successfully!


## Implementation Overview

The producer consists of three main functions:

1. **`load_batches(filepath)`** — reads the CSV, groups rows by `batch_id`, converts fields to correct types, strips whitespace from `car_plate`, and skips malformed rows safely with a warning log.

2. **`connect_producer(broker)`** — creates and connects a JSON-serialised `KafkaProducer` to the Kafka broker.

3. **`publish_batches(producer, batches)`** — publishes one batch at a time, stamps `produced_at` and `batch_size` on every event, flushes messages to ensure broker acknowledgement, logs the next checkpoint batch_id for restart purposes, and sleeps between batches to simulate streaming.

For demonstration purposes, the producer may be stopped after ~10–15 batches since full dataset completion is not required during live demo execution.

In [3]:
HOST_IP = '192.168.0.175'
KAFKA_BROKER = '192.168.0.175:9092'   # use container IP directly, not HOST_IP
TOPIC = 'camera-events-A'
PRODUCER_ID = 'A'
DATA_FILE = '../data/camera_event_A.csv'
BATCH_SLEEP_SECS = 2


In [4]:
def load_batches(filepath):
    """
    Read camera event CSV and group rows by batch_id.

    Args:
        filepath (str): Path to the camera event CSV file.

    Returns:
        list: Sorted list of (batch_id, [events]) tuples ordered by batch_id.
    """
    batches = defaultdict(list)
    skipped = 0

    with open(filepath, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                event = {
                    'event_id' : str(row['event_id']),
                    'batch_id'  : int(row['batch_id']),
                    'car_plate' : row['car_plate'].strip(),
                    'camera_id' : int(row['camera_id']),
                    'timestamp' : row['timestamp'],
                    'speed_reading': round(float(row['speed_reading']), 2),
                    'producer_id' : PRODUCER_ID
                }
                batches[event['batch_id']].append(event)
            except (KeyError, ValueError) as e:
                skipped += 1
                print(f'[WARN] Skipping malformed row: {e} | row={row}')

    total_events = sum(len(v) for v in batches.values())
    print(f'[INFO] Loaded {total_events} events across {len(batches)} batches. '
          f'Skipped {skipped} malformed rows.')
    return sorted(batches.items())

In [5]:
def connect_producer(broker):
    """
    Create and connect a JSON-serialised KafkaProducer to the Kafka broker.

    Args:
        broker (str): Kafka broker address in host:port format.

    Returns:
        KafkaProducer: Connected producer instance ready to publish messages.
    """
    producer = KafkaProducer(
        bootstrap_servers=[broker],
        value_serializer=lambda v: json.dumps(v).encode('utf-8'),
        api_version=(2, 5, 0),          # Matches your container API version
        api_version_auto_timeout_ms=5000 # Prevents the notebook from hanging if a packet drops
    )
    print(f'[INFO] Producer {PRODUCER_ID} connected → {broker} | topic: {TOPIC}')
    return producer

In [6]:
def publish_batches(producer, batches):
    """
    Publish all batches to the Kafka topic one batch at a time.

    Per batch: stamps produced_at and batch_size on each event,
    sends all events, flushes, then sleeps before next batch.
    Logs checkpoint (next batch_id) so restarts are posible after interruption.

    Args:
        producer (KafkaProducer): Connected Kafka producer.
        batches (list): Sorted list of (batch_id, [events]) tuples.
    """
    total_sent = 0

    for batch_id, events in batches:
        produced_at = datetime.now(timezone.utc).isoformat()
        batch_size  = len(events)
        batch_sent  = 0

        for event in events:
            event['produced_at'] = produced_at
            event['batch_size']  = batch_size
            try:
                producer.send(TOPIC, value=event)
                batch_sent += 1
            except KafkaError as e:
                print(f'[ERROR] Failed to send event {event["event_id"]}: {e}')

        producer.flush()
        total_sent += batch_sent
        print(f'[batch_id={batch_id:>6}] sent={batch_sent:>3} | '
              f'total={total_sent:>6} | next checkpoint: batch_id={batch_id + 1}')
        time.sleep(BATCH_SLEEP_SECS)

    print(f'[INFO] Producer A complete. Total sent: {total_sent}')

## Execution Notes:

* Producers A, B, and C should be executed **concurrently in separate Jupyter notebook tabs** during the demonstration to ensure that Spark receives event streams from all three Kafka topics simultaneously.
* If the network connection changes (for example, when reconnecting to a mobile hotspot or switching Wi-Fi networks), update the `HOST_IP` variable before re-running the system, as the Kafka broker address is dependent on the machine’s current IPv4 address.
* If a `NoBrokersAvailable` error occurs, verify that the Kafka container is active by running `docker ps`, and ensure that the configured IP address matches the output provided by `ipconfig`.


In [ ]:
# Main:
batches = load_batches(DATA_FILE)
producer = connect_producer(KAFKA_BROKER)

try:
    publish_batches(producer, batches)
except KeyboardInterrupt:
    print('[INFO] Producer A interrupted by user.')
finally:
    producer.close()
    print('[INFO] Producer A connection closed.')

[INFO] Loaded 556340 events across 27817 batches. Skipped 0 malformed rows.
[INFO] Producer A connected → 192.168.0.175:9092 | topic: camera-events-A
[batch_id=     1] sent= 20 | total=    20 | next checkpoint: batch_id=2
[batch_id=     2] sent= 20 | total=    40 | next checkpoint: batch_id=3
[batch_id=     3] sent= 20 | total=    60 | next checkpoint: batch_id=4
[batch_id=     4] sent= 20 | total=    80 | next checkpoint: batch_id=5
[batch_id=     5] sent= 20 | total=   100 | next checkpoint: batch_id=6
[batch_id=     6] sent= 20 | total=   120 | next checkpoint: batch_id=7
[batch_id=     7] sent= 20 | total=   140 | next checkpoint: batch_id=8
[batch_id=     8] sent= 20 | total=   160 | next checkpoint: batch_id=9
[batch_id=     9] sent= 20 | total=   180 | next checkpoint: batch_id=10
[batch_id=    10] sent= 20 | total=   200 | next checkpoint: batch_id=11
[batch_id=    11] sent= 20 | total=   220 | next checkpoint: batch_id=12
[batch_id=    12] sent= 20 | total=   240 | next checkp

[batch_id=   112] sent= 20 | total=  2240 | next checkpoint: batch_id=113
[batch_id=   113] sent= 20 | total=  2260 | next checkpoint: batch_id=114
[batch_id=   114] sent= 20 | total=  2280 | next checkpoint: batch_id=115
[batch_id=   115] sent= 20 | total=  2300 | next checkpoint: batch_id=116
[batch_id=   116] sent= 20 | total=  2320 | next checkpoint: batch_id=117
[batch_id=   117] sent= 20 | total=  2340 | next checkpoint: batch_id=118
[batch_id=   118] sent= 20 | total=  2360 | next checkpoint: batch_id=119
[batch_id=   119] sent= 20 | total=  2380 | next checkpoint: batch_id=120
[batch_id=   120] sent= 20 | total=  2400 | next checkpoint: batch_id=121
[batch_id=   121] sent= 20 | total=  2420 | next checkpoint: batch_id=122
[batch_id=   122] sent= 20 | total=  2440 | next checkpoint: batch_id=123
[batch_id=   123] sent= 20 | total=  2460 | next checkpoint: batch_id=124
[batch_id=   124] sent= 20 | total=  2480 | next checkpoint: batch_id=125
[batch_id=   125] sent= 20 | total=  2

[batch_id=   223] sent= 20 | total=  4460 | next checkpoint: batch_id=224
[batch_id=   224] sent= 20 | total=  4480 | next checkpoint: batch_id=225
[batch_id=   225] sent= 20 | total=  4500 | next checkpoint: batch_id=226
[batch_id=   226] sent= 20 | total=  4520 | next checkpoint: batch_id=227
[batch_id=   227] sent= 20 | total=  4540 | next checkpoint: batch_id=228
[batch_id=   228] sent= 20 | total=  4560 | next checkpoint: batch_id=229
[batch_id=   229] sent= 20 | total=  4580 | next checkpoint: batch_id=230
[batch_id=   230] sent= 20 | total=  4600 | next checkpoint: batch_id=231
[batch_id=   231] sent= 20 | total=  4620 | next checkpoint: batch_id=232
[batch_id=   232] sent= 20 | total=  4640 | next checkpoint: batch_id=233
[batch_id=   233] sent= 20 | total=  4660 | next checkpoint: batch_id=234
[batch_id=   234] sent= 20 | total=  4680 | next checkpoint: batch_id=235
[batch_id=   235] sent= 20 | total=  4700 | next checkpoint: batch_id=236
[batch_id=   236] sent= 20 | total=  4

[batch_id=   334] sent= 20 | total=  6680 | next checkpoint: batch_id=335
[batch_id=   335] sent= 20 | total=  6700 | next checkpoint: batch_id=336
[batch_id=   336] sent= 20 | total=  6720 | next checkpoint: batch_id=337
[batch_id=   337] sent= 20 | total=  6740 | next checkpoint: batch_id=338
[batch_id=   338] sent= 20 | total=  6760 | next checkpoint: batch_id=339
[batch_id=   339] sent= 20 | total=  6780 | next checkpoint: batch_id=340
[batch_id=   340] sent= 20 | total=  6800 | next checkpoint: batch_id=341
[batch_id=   341] sent= 20 | total=  6820 | next checkpoint: batch_id=342
[batch_id=   342] sent= 20 | total=  6840 | next checkpoint: batch_id=343
[batch_id=   343] sent= 20 | total=  6860 | next checkpoint: batch_id=344
[batch_id=   344] sent= 20 | total=  6880 | next checkpoint: batch_id=345
[batch_id=   345] sent= 20 | total=  6900 | next checkpoint: batch_id=346
[batch_id=   346] sent= 20 | total=  6920 | next checkpoint: batch_id=347
[batch_id=   347] sent= 20 | total=  6

[batch_id=   445] sent= 20 | total=  8900 | next checkpoint: batch_id=446
[batch_id=   446] sent= 20 | total=  8920 | next checkpoint: batch_id=447
[batch_id=   447] sent= 20 | total=  8940 | next checkpoint: batch_id=448
[batch_id=   448] sent= 20 | total=  8960 | next checkpoint: batch_id=449
[batch_id=   449] sent= 20 | total=  8980 | next checkpoint: batch_id=450
[batch_id=   450] sent= 20 | total=  9000 | next checkpoint: batch_id=451
[batch_id=   451] sent= 20 | total=  9020 | next checkpoint: batch_id=452
[batch_id=   452] sent= 20 | total=  9040 | next checkpoint: batch_id=453
[batch_id=   453] sent= 20 | total=  9060 | next checkpoint: batch_id=454
[batch_id=   454] sent= 20 | total=  9080 | next checkpoint: batch_id=455
[batch_id=   455] sent= 20 | total=  9100 | next checkpoint: batch_id=456
[batch_id=   456] sent= 20 | total=  9120 | next checkpoint: batch_id=457
[batch_id=   457] sent= 20 | total=  9140 | next checkpoint: batch_id=458
[batch_id=   458] sent= 20 | total=  9

...

[batch_id=  3640] sent= 20 | total= 72800 | next checkpoint: batch_id=3641
[batch_id=  3641] sent= 20 | total= 72820 | next checkpoint: batch_id=3642
[batch_id=  3642] sent= 20 | total= 72840 | next checkpoint: batch_id=3643
[batch_id=  3643] sent= 20 | total= 72860 | next checkpoint: batch_id=3644
[batch_id=  3644] sent= 20 | total= 72880 | next checkpoint: batch_id=3645
[batch_id=  3645] sent= 20 | total= 72900 | next checkpoint: batch_id=3646
[batch_id=  3646] sent= 20 | total= 72920 | next checkpoint: batch_id=3647
[batch_id=  3647] sent= 20 | total= 72940 | next checkpoint: batch_id=3648
[batch_id=  3648] sent= 20 | total= 72960 | next checkpoint: batch_id=3649
[batch_id=  3649] sent= 20 | total= 72980 | next checkpoint: batch_id=3650
[batch_id=  3650] sent= 20 | total= 73000 | next checkpoint: batch_id=3651
[batch_id=  3651] sent= 20 | total= 73020 | next checkpoint: batch_id=3652
[batch_id=  3652] sent= 20 | total= 73040 | next checkpoint: batch_id=3653
[batch_id=  3653] sent= 2

[batch_id=  3750] sent= 20 | total= 75000 | next checkpoint: batch_id=3751
[batch_id=  3751] sent= 20 | total= 75020 | next checkpoint: batch_id=3752
[batch_id=  3752] sent= 20 | total= 75040 | next checkpoint: batch_id=3753
[batch_id=  3753] sent= 20 | total= 75060 | next checkpoint: batch_id=3754
[batch_id=  3754] sent= 20 | total= 75080 | next checkpoint: batch_id=3755
[batch_id=  3755] sent= 20 | total= 75100 | next checkpoint: batch_id=3756
[batch_id=  3756] sent= 20 | total= 75120 | next checkpoint: batch_id=3757
[batch_id=  3757] sent= 20 | total= 75140 | next checkpoint: batch_id=3758
[batch_id=  3758] sent= 20 | total= 75160 | next checkpoint: batch_id=3759
[batch_id=  3759] sent= 20 | total= 75180 | next checkpoint: batch_id=3760
[batch_id=  3760] sent= 20 | total= 75200 | next checkpoint: batch_id=3761
[batch_id=  3761] sent= 20 | total= 75220 | next checkpoint: batch_id=3762
[batch_id=  3762] sent= 20 | total= 75240 | next checkpoint: batch_id=3763
[batch_id=  3763] sent= 2

[batch_id=  3860] sent= 20 | total= 77200 | next checkpoint: batch_id=3861
[batch_id=  3861] sent= 20 | total= 77220 | next checkpoint: batch_id=3862
[batch_id=  3862] sent= 20 | total= 77240 | next checkpoint: batch_id=3863
[batch_id=  3863] sent= 20 | total= 77260 | next checkpoint: batch_id=3864
[batch_id=  3864] sent= 20 | total= 77280 | next checkpoint: batch_id=3865
[batch_id=  3865] sent= 20 | total= 77300 | next checkpoint: batch_id=3866
[batch_id=  3866] sent= 20 | total= 77320 | next checkpoint: batch_id=3867
[batch_id=  3867] sent= 20 | total= 77340 | next checkpoint: batch_id=3868
[batch_id=  3868] sent= 20 | total= 77360 | next checkpoint: batch_id=3869
[batch_id=  3869] sent= 20 | total= 77380 | next checkpoint: batch_id=3870
[batch_id=  3870] sent= 20 | total= 77400 | next checkpoint: batch_id=3871
[batch_id=  3871] sent= 20 | total= 77420 | next checkpoint: batch_id=3872
[batch_id=  3872] sent= 20 | total= 77440 | next checkpoint: batch_id=3873
[batch_id=  3873] sent= 2

[batch_id=  3970] sent= 20 | total= 79400 | next checkpoint: batch_id=3971
[batch_id=  3971] sent= 20 | total= 79420 | next checkpoint: batch_id=3972
[batch_id=  3972] sent= 20 | total= 79440 | next checkpoint: batch_id=3973
[batch_id=  3973] sent= 20 | total= 79460 | next checkpoint: batch_id=3974
[batch_id=  3974] sent= 20 | total= 79480 | next checkpoint: batch_id=3975
[batch_id=  3975] sent= 20 | total= 79500 | next checkpoint: batch_id=3976
[batch_id=  3976] sent= 20 | total= 79520 | next checkpoint: batch_id=3977
[batch_id=  3977] sent= 20 | total= 79540 | next checkpoint: batch_id=3978
[batch_id=  3978] sent= 20 | total= 79560 | next checkpoint: batch_id=3979
[batch_id=  3979] sent= 20 | total= 79580 | next checkpoint: batch_id=3980
[batch_id=  3980] sent= 20 | total= 79600 | next checkpoint: batch_id=3981
[batch_id=  3981] sent= 20 | total= 79620 | next checkpoint: batch_id=3982
[batch_id=  3982] sent= 20 | total= 79640 | next checkpoint: batch_id=3983
[batch_id=  3983] sent= 2

[batch_id=  4080] sent= 20 | total= 81600 | next checkpoint: batch_id=4081
[batch_id=  4081] sent= 20 | total= 81620 | next checkpoint: batch_id=4082
[batch_id=  4082] sent= 20 | total= 81640 | next checkpoint: batch_id=4083
[batch_id=  4083] sent= 20 | total= 81660 | next checkpoint: batch_id=4084
[batch_id=  4084] sent= 20 | total= 81680 | next checkpoint: batch_id=4085
[batch_id=  4085] sent= 20 | total= 81700 | next checkpoint: batch_id=4086
[batch_id=  4086] sent= 20 | total= 81720 | next checkpoint: batch_id=4087
[batch_id=  4087] sent= 20 | total= 81740 | next checkpoint: batch_id=4088
[batch_id=  4088] sent= 20 | total= 81760 | next checkpoint: batch_id=4089
[batch_id=  4089] sent= 20 | total= 81780 | next checkpoint: batch_id=4090
[batch_id=  4090] sent= 20 | total= 81800 | next checkpoint: batch_id=4091
[batch_id=  4091] sent= 20 | total= 81820 | next checkpoint: batch_id=4092
[batch_id=  4092] sent= 20 | total= 81840 | next checkpoint: batch_id=4093
[batch_id=  4093] sent= 2